
## 1. Delta Lake Fundamentals


## Q1. What is Delta Lake?


- **Delta Lake** is an `open-source storage framework` built on Parquet files. 
- It provides `ACID properties, schema enforcement, schema evolution, time travel etc...` .
- It used to build `reliable data pipelines` on top of a data lake.


## Q2. What is the difference between Data Lake, Delta Lake and Lakehouse?


| Concept | Explanation | Example |
|---|---|---|
| Data Lake | Stores raw structured and unstructured data | CSV, JSON, images, Parquet |
| Delta Lake | Adds transaction and reliability capabilities to data lake storage | Delta tables |
| Lakehouse | Combines flexibility of data lake with capability of data warehouse | Databricks Lakehouse |


## Q3. What are the components of a Delta table?


**A Delta table mainly consists of:**

- Parquet files.
- transaction log.
- Checkpoint files.
- Deletion vector files


| Component | Purpose |
|---|---|
| Parquet files | Store actual table data |
| JSON transaction log | Records committed table changes |
| Checkpoint | Stores a consolidated snapshot of transaction state |
| Deletion vectors | Identify deleted or updated rows without immediately rewriting entire Parquet files |


## Q4. What is the difference between Parquet, ORC and Delta?


| Feature | Parquet | ORC | Delta |
|---|---|---|---|
| Type | Columnar file format | Columnar file format | Table/storage framework |
| ACID properties | not performed | not performed | performed |
| Time Travel | not performed | not performed | performed |
| Schema enforcement | not Supported | not Supported | Supported |
| Schema evolution | not Supported | not Supported | Supported |


## Q5. What are ACID properties in Delta Lake?


ACID stands for:

- A – Atomicity
- C – Consistency
- I – Isolation
- D – Durability


| Property | Meaning | Explanation |
|---|---|---|
| **Atomicity** | All or Nothing | A transaction must either complete entirely or fail entirely. |
| **Consistency** | Valid Data | Every successful transaction must leave the Delta table in a valid state. |
| **Isolation** | Independent Transactions | Multiple transactions can execute concurrently without exposing partially committed changes to readers. |
| **Durability** | Permanent Changes | Once a transaction is successfully committed, its changes are persisted. |



## Q6. What is the Delta transaction log?


- The **Delta Transaction Log** is a `JSON-based transaction file` that records `every change in a Delta table`.
- It is stored in the `_delta_log directory` inside the **Delta table's storage location**.
- It helps **Delta Lake** maintain `ACID transactions, versioning, and time travel`.


## Q7. What is a checkpoint in Delta Lake?


- A **checkpoint** is a `consolidated snapshot of the transaction log state`. 
- It is used to `improve query performance`.
- **Delta Lake** stores `checkpoints` in `Parquet format` inside the **_delta_log directory.**


## Q8. What are deletion vectors?


- **Deletion vectors** are used to **identify** are `deleted or updated rows` without immediately rewriting the `complete Parquet file`.
- They can **improve the performance** of `DELETE, UPDATE and MERGE` operations.


## Q9. What is schema enforcement?


- **Schema enforcement** is used to check the `incoming data schema` with `target Delta table's schema`.
- It checks `column names, data types` and `schema compatibility`.
- It helps to maintain `data quality` and `consistency`.


**Example:**


```python
orders_df.write\
    .format("delta")\
        .mode("append")\
            .saveAsTable("workspace.ecommerce_silver.orders")


## Q10. What is schema evolution?


- **Schema evolution** is used to `change the Delta table schema` when **new data contains additional columns**.
- It can **automatically** `add new columns`.
- **mergeSchema = true** enables `schema evolution`.
- It reduces `manual schema modification`.


## Q11. What is Time Travel?


- **Time Travel** is used to `access previous versions of a Delta table`.
- Performs **Time Travel** using `Version Number`,`Timestamp`, `SQL`.
- It uses the `Delta transaction log (_delta_log)`.
- We can access **historical data** using `versionAsOf` and `timestampAsOf`.
- `DESCRIBE HISTORY` displays **table history**.


## Q12. What is MERGE in Delta Lake?


- **MERGE** is used to perform `INSERT, UPDATE, and DELETE` operations in a **single transaction** by `source and target` tables.
- It is mainly used for `incremental data pipelines` and `CDC processing`.


### 1. Existing Customer Table

| customer_id | name | city |
|---|---|---|
| 101 | Arun | Chennai |


### 2. Incoming Source Data

| customer_id | name | city |
|---|---|---|
| 101 | Arun Kumar | Bangalore |
| 102 | Priya | Madurai |


### 3. After MERGE Operation

| customer_id | name | city |
|---|---|---|
| 101 | Arun Kumar | Bangalore |
| 102 | Priya | Madurai |


## Q13. What is the difference between INSERT, UPDATE, DELETE and UPSERT?


## DML Operations:


| Operation | Purpose | Example |
|---|---|---|
| INSERT | Adds new records | New customer |
| UPDATE | Modifies existing records | Customer changes city |
| DELETE | Removes matching records | Invalid customer |
| UPSERT | Updates existing and inserts new records | Daily customer feed |


## Q14. What happens if multiple source records match one target record in MERGE?


- If multiple source records match the same target records in merge, then **MERGE can fail because the update is ambiguous**.
- We should `deduplicate the source data` before **performing MERGE**.


**Example:**


```python
from pyspark.sql.window import Window 
from pyspark.sql.functions import row_number, col

window_spec = (Window.partitionBy("customer_id")    
                     .orderBy(col("updated_at")
                     .desc()))
                     
deduplicated_df = (source_df.withColumn("rn", row_number().over(window_spec))
.filter(col("rn") == 1)
.drop("rn"))


## Q15. What is SCD?


**SCD (Slowly Changing Dimension)** is a technique in `data warehousing` to **manage and track changes** in `dimension table data` over time.


## Q16. What is SCD Type 1?


- Overwrites `existing records with new values`.
- **Does not** maintain historical data.
- Update the `existing records`.
- **Previous values** are permanently replaced.
- Requires **less storag**e.
- **Simple** to implement.


## Q17. What is SCD Type 2?


- Inserts a `new record` whenever a **tracked attribute changes**.
- Maintains `complete historical data`.
- Creates a `new record`.
- Previous values for `historical analysis`.
- Requires **more storage**.
- **More complex** to implement.


## Q18. Difference between SCD Type 1 and Type 2?


**SCD Type 1:**

- Overwrites `existing records with new values`.
- **Does not** maintain historical data.
- Update the `existing records`.
- **Previous values** are permanently replaced.
- Requires **less storag**e.
- **Simple** to implement.

**SCD Type 2:**

- Inserts a `new record` whenever a **tracked attribute changes**.
- Maintains `complete historical data`.
- Creates a `new record`.
- Previous values for `historical analysis`.
- Requires **more storage**.
- **More complex** to implement.


## Q19. What is OPTIMIZE in Delta Lake?


- **OPTIMIZE** is used to `improve Delta table performance` by **reorganizing data files** and **reducing small-file problems**.
- It performs `bin-packing` by combining **smaller files** into **larger**, **more efficient files**.


## Q20. What is ZORDER in Delta Lake?


- **ZORDER** is a `data-layout optimization technique` that **colocates related data values in the same set of files**.
- It improves `data skipping`, reducing the `number of files` that Spark **needs to scan.**


## Q21. What is Partitioning in Delta Lake?


- **Partitioning** is a technique used to divide a `large Delta table` into `smaller parts` based on **one or more columns**.
- It improves **query performance** by `reducing the amount of data scanned`.



## Q22. Difference between OPTIMIZE, ZORDER and Partitioning?


### OPTIMIZE:

- **OPTIMIZE** is used to `improve Delta table performance` by **reorganizing data files** and **reducing small-file problems**.
- It performs `bin-packing` by combining **smaller files** into **larger**, **more efficient files**.

### ZORDER:

- **ZORDER** is a `data-layout optimization technique` that **colocates related data values in the same set of files**.
- It improves `data skipping`, reducing the `number of files` that Spark **needs to scan.**

### Partitioning:

- **Partitioning** is a technique used to divide a `large Delta table` into `smaller parts` based on **one or more columns**.
- It improves **query performance** by `reducing the amount of data scanned`.


## Q23. What is Liquid Clustering?


- **Liquid Clustering** is a `data optimization technique in Delta Lake`.
- It is used to `organizes data in a table` by selected `clustering columns` to improve `query performance and data skipping`.



**Create a Liquid Clustered table:**


```python
CREATE TABLE workspace.ecommerce_silver.orders_liquid
USING DELTA
CLUSTER BY (customer_id, order_date)
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


**Enable Liquid Clustering on an existing table:**


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);


**Then reorganize the existing data:**


```python
OPTIMIZE workspace.ecommerce_silver.orders FULL;


OPTIMIZE FULL is supported in Databricks Runtime 16.0 and above.


## Q24. What are the advantages of Liquid Clustering?


- Improved Query Performance
- Better Data Skipping
- Flexible Clustering
- Incremental Clustering
- Reduced Small-File Problems
- Supports Large Tables


### Change clustering keys:


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (product_id, customer_id);


### Recluster existing data:


```python
OPTIMIZE workspace.ecommerce_silver.orders FULL;


### Q1. Is Liquid Clustering a replacement for ZORDER?


For many new Delta tables, yes, it is the recommended alternative. However, ZORDER remains supported for existing workloads.


### Q2. Does Liquid Clustering require partitioning?


No. Liquid Clustering works without traditional partitioning.


### Q3. Does Liquid Clustering automatically organize data?


Databricks supports automatic clustering through predictive optimization for eligible tables and configurations. Otherwise, you can trigger clustering using OPTIMIZE.


### Q4. Can we change clustering columns?


Yes. 

We can change clustering columns using ALTER TABLE ... CLUSTER BY.


## Q25. Difference between Z-Ordering and Liquid Clustering?


### ZORDER:

- **ZORDER** is a `data-layout optimization technique` that **colocates related data values in the same set of files**.
- It improves `data skipping`, reducing the `number of files` that Spark **needs to scan.**

**Example:**

```python
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);
```

### Liquid Clustering:

- **Liquid Clustering** is a `data optimization technique in Delta Lake`.
- It is used to `organizes data in a table` by selected `clustering columns` to improve `query performance and data skipping`.

**Example:**

```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);

OPTIMIZE workspace.ecommerce_silver.orders;


## Q26. How do you convert an existing Z-Ordering table to Liquid Clustering?


**Suppose our existing table uses Z-Ordering:**


```python
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);


We want to migrate it to Liquid Clustering.


### Step 1: Enable Liquid Clustering


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);


### Step 2: Recluster existing files


```python
OPTIMIZE workspace.ecommerce_silver.orders FULL;


### Step 3: Verify table configuration


```python
DESCRIBE DETAIL workspace.ecommerce_silver.orders;


## Q27. What is the Difference Between CLUSTER BY and partitionBy?


- **partitionBy** divides `data` into `separate physical directories` based on **column values** and `improves query performance through partition pruning`.
- **CLUSTER BY** is a **Liquid Clustering** is a `data optimization technique in Delta Lake`.
- It is used to `organizes data in a table` by selected `clustering columns` to improve `query performance and data skipping`.


### Liquid Clustering:


```python
CREATE TABLE workspace.ecommerce_silver.orders_clustered
USING DELTA
CLUSTER BY (customer_id, order_date)
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


## Q28. What is VACUUM in Delta Lake?


- **VACUUM** is a `Delta Lake command` used to `remove old, unreferenced data files`.
- that are `no longer required by the Delta table`, helping to **free up storage space**.


**Example:**


```python
VACUUM workspace.ecommerce_silver.orders;


**Vacuum with a specified retention period:**


```python
VACUUM workspace.ecommerce_silver.orders
RETAIN 168 HOURS;


### Important interview questions:


**1. Does VACUUM delete the latest active data files?**

- No. 
- It removes obsolete files that are eligible for deletion under the retention policy.


**2. Does VACUUM delete _delta_log JSON files?**

- No. 
- Transaction-log cleanup is handled separately through Delta Lake log retention and checkpoint processing.


**3. What happens if we run VACUUM with zero hours?**

- It can delete files required by concurrent operations and older table versions.
- Databricks has a safety check against unsafe retention periods. Disabling that protection without understanding the consequences can cause data loss.


**4. What is the default retention of the transaction log?**

- **Deleted data-file retention:** 7 days.
- **Transaction-log retention:** 30 days.
- These are separate settings.


| Feature | Time Travel | VACUUM |
|---|---|---|
| Purpose | Reads previous table versions | Removes eligible obsolete data files |
| Main Function | Accesses historical table data | Deletes old, unreferenced data files |
| Usage | Supports historical analysis and recovery | Reduces storage usage |
| Data Access | Uses retained data files and transaction logs | Identifies and removes eligible obsolete files |
| Operation | Reads historical snapshots | Physically deletes eligible files |
| Historical Data | Allows access to previous versions | Can make older versions unreadable |
| Command | VERSION AS OF / TIMESTAMP AS OF | VACUUM table_name |
| Main Benefit | Historical data recovery | Storage management |


## Q29. What is CTAS in Databricks?


- **CTAS** stands for `CREATE TABLE AS SELECT`.
- It **creates and populates** a new table using the results of a SELECT query.


### Example:


```python
CREATE TABLE workspace.ecommerce_silver.orders_ctas
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


## Q30. What is DEEP CLONE?


- **DEEP CLONE** is used to **creates** an `copy of a Delta table`, including its `data files` and `supported table metadata`.
- The **cloned table** can be `modified` independently `without affecting` the **source table**.


### Example:


```python
CREATE TABLE workspace.ecommerce_silver.orders_deep_clone
DEEP CLONE workspace.ecommerce_silver.orders;


## Q31. What is SHALLOW CLONE?


- **SHALLOW CLONE** is used to creates a `new Delta table` that copies `table metadata`.
- It requires `less storage` and compute than `deep cloning`.


### Example:


```python
CREATE TABLE workspace.ecommerce_silver.orders_shallow_clone
SHALLOW CLONE workspace.ecommerce_silver.orders;


## Q32. Difference Between CTAS, DEEP CLONE and SHALLOW CLONE?


### CTAS:

- **CTAS** stands for `CREATE TABLE AS SELECT`.
- It **creates and populates** a new table using the results of a SELECT query.

### DEEP CLONE:

- **DEEP CLONE** is used to **creates** an `copy of a Delta table`, including its `data files` and `supported table metadata`.
- The **cloned table** can be `modified` independently `without affecting` the **source table**.

### SHALLOW CLONE:

- **SHALLOW CLONE** is used to creates a `new Delta table` that copies `table metadata`.
- It requires `less storage` and compute than `deep cloning`.


## Q33. Hands-on: Create Tables Using CTAS, DEEP CLONE and SHALLOW CLONE and Compare Storage and Metadata.


### Step 1. Create the Source Table:


```sql
CREATE TABLE workspace.ecommerce_silver.orders_source
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders;
```



### Step 2. Create a Table Using CTAS:



```sql
CREATE TABLE workspace.ecommerce_silver.orders_ctas
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders_source;
```


### Step 3. Create a Table Using DEEP CLONE:


```sql
CREATE TABLE workspace.ecommerce_silver.orders_deep
DEEP CLONE workspace.ecommerce_silver.orders_source;
```


### Step 4. Create a Table Using SHALLOW CLONE:


```sql
CREATE TABLE workspace.ecommerce_silver.orders_shallow
SHALLOW CLONE workspace.ecommerce_silver.orders_source;
```


### Step 5. Compare Row Counts:


```sql
SELECT
    'SOURCE' AS table_name,
    COUNT(*) AS total_rows
FROM workspace.ecommerce_silver.orders_source

UNION ALL

SELECT
    'CTAS',
    COUNT(*)
FROM workspace.ecommerce_silver.orders_ctas

UNION ALL

SELECT
    'DEEP CLONE',
    COUNT(*)
FROM workspace.ecommerce_silver.orders_deep

UNION ALL

SELECT
    'SHALLOW CLONE',
    COUNT(*)
FROM workspace.ecommerce_silver.orders_shallow;
```



### Step 6. Compare Table Metadata:


**Source Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_source;
```


**CTAS Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_ctas;
```


**DEEP CLONE Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_deep;
```


**SHALLOW CLONE Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_shallow;
```


### Step 7. Compare Delta Table History:


**Source Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_source;
```


**CTAS Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_ctas;
```


**DEEP CLONE Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_deep;
```


**SHALLOW CLONE Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_shallow;
```


## Q34. What is Change Data Feed (CDF) in Delta Lake?


- **Change Data Feed (CDF)** is a `Delta Lake feature` is used to captures `row-level changes made to a Delta table`, including **INSERT, UPDATE and DELETE operations**.
- It allows **downstream pipelines** to process `only changed records` instead of **reading the entire source table**.



## Q35. How do you enable Change Data Feed?


### Step 1: Enable CDF:


```sql
ALTER TABLE workspace.ecommerce_silver.customers
SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);
```


### Step 2: Insert a record:


```sql
INSERT INTO workspace.ecommerce_silver.customers
VALUES (103, 'Rahul', 'Coimbatore');
```


### Step 3: Update a record:


```sql
UPDATE workspace.ecommerce_silver.customers
SET city = 'Bangalore'
WHERE customer_id = 101;
```


### Step 4: Delete a record:


```sql
DELETE FROM workspace.ecommerce_silver.customers
WHERE customer_id = 102;
```


### Step 5: Read CDF changes:


```sql
cdf_df = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("workspace.ecommerce_silver.customers")
)

display(cdf_df)
```


## Q36. How do you use CDF in a real-time incremental pipeline?


### Step 1: Read changes:


```sql
cdf_df = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 5)
    .table("workspace.ecommerce_silver.customers")
)

display(cdf_df)
```


### Step 2: Filter inserted and updated records


```sql
from pyspark.sql.functions import col

changed_records = cdf_df.filter(
    col("_change_type").isin(
        "insert",
        "update_postimage"
    )
)
```



### Step 3: Read deleted records:


```sql
deleted_records = cdf_df.filter(
    col("_change_type") == "delete"
)
```


### Step 4: Write all changes into an audit table:


```sql
(
    cdf_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "workspace.ecommerce_silver.customer_audit"
    )
)
```


## Q37. What is the difference between CDF and Time Travel in Delta Lake?


| Feature | CDF (Change Data Feed) | Time Travel |
|---|---|---|
| Purpose | Captures row-level changes between table versions | Reads previous versions of a table |
| Output | Returns changed records | Returns the complete table snapshot |
| Insert detection | Yes | Not directly |
| Update detection | Yes | Not directly |
| Delete detection | Yes | Not directly |
| Main use | Incremental ETL, CDC, auditing | Historical queries, data recovery |


## Q38. What happens when an UPDATE operation runs in Delta Lake?


**When an UPDATE operation runs:**

- Delta Lake identifies the affected records.
- It creates a new committed table version.
- Depending on the execution strategy and table features, it may rewrite affected data files or use deletion vectors.
- The transaction log records the committed changes.
- Old data files may remain available for historical queries until retention cleanup.



**Example:**


```sql
UPDATE workspace.ecommerce_silver.orders
SET order_status = 'COMPLETED'
WHERE order_id = 'O101';
```


## Q39. Can we use UPDATE and DELETE in Parquet?


- **Parquet** is a `columnar file format`. 
- It does not provide native transactional UPDATE and DELETE operations.
- To modify Parquet data, we generally need to read the existing data, apply transformations and rewrite the affected files or dataset.
- Delta Lake provides transactional UPDATE, DELETE and MERGE operations on top of data files.


**Example:**


```sql
UPDATE workspace.ecommerce_silver.orders
SET order_status = 'CANCELLED'
WHERE order_id = 'O101';
```


## Q40. What is idempotency in Delta Lake?


- **Idempotency** is used to executing `the same data pipeline multiple times produces the same result` without creating duplicate records or changing the result incorrectly.
- In Delta Lake, idempotency is important when a pipeline fails midway and needs to be restarted.


**Example:**


```sql
MERGE INTO workspace.ecommerce_silver.orders AS target
USING workspace.ecommerce_bronze.orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;
```


## Q41. Can multiple users access a Delta table simultaneously?


- Yes.
- Delta Lake supports concurrent reads and writes through its transaction protocol and concurrency-control mechanisms.


## Q42. What is Data Skipping?


- **Data Skipping** is a `Delta Lake optimization technique` that avoids reading `unnecessary data files` during query execution.
- Delta Lake collects statistics such as minimum value, maximum value, and null count for columns in data files.
- When a query contains filter conditions, Spark uses these statistics to identify which files may contain matching records and skips files that definitely cannot contain them.
- This reduces disk I/O and improves query performance.


## Q43. What is the difference between OPTIMIZE and VACUUM in Delta Lake?


### OPTIMIZE:

- **OPTIMIZE** is used to `improve Delta table performance` by **reorganizing data files** and **reducing small-file problems**.
- It performs `bin-packing` by combining **smaller files** into **larger**, **more efficient files**.

### VACUUM:

- **VACUUM** is a `Delta Lake command` used to `remove old, unreferenced data files`.
- that are `no longer required by the Delta table`, helping to **free up storage space**.


## Q44. What is the difference between Delta Table History and Time Travel?


### Delta Table History:

- **Delta Table History** is used to `track all operations performed on a Delta table`.
- It provides details such as `operation type, timestamp, user, and operation parameters`.
- It helps track `INSERT, UPDATE, DELETE, MERGE, and OPTIMIZE` operations.
- It is used to `identify previous versions of a Delta table`.
- We can view table history using the `DESCRIBE HISTORY` command.

### Time Travel:

- **Time Travel** is used to `access previous versions of a Delta table`.
- Performs **Time Travel** using `Version Number`,`Timestamp`, `SQL`.
- It uses the `Delta transaction log (_delta_log)`.
- We can access **historical data** using `versionAsOf` and `timestampAsOf`.
- `DESCRIBE HISTORY` displays **table history**.


## Q45. Can we recover deleted data after VACUUM?



If VACUUM permanently deletes the data files required for an older table version, Delta Time Travel cannot reconstruct those records from the transaction log alone.


**Recovery requires another available source, such as:**

- Backup.
- Deep clone
- Archived data.
- Replicated copy.
- External recovery mechanism.